In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib
from pyspark.sql import functions as F

In [ ]:
ambiente = 'dev'

In [ ]:
data = (
    spark.sql(
        f"""
            SELECT coast_name, datetime, wind_u, wind_v, wave_u, wave_v, wave_period_s
            FROM cor_{ambiente}.silver.swell_metrics
        """
    )
)

In [ ]:
data_pre_processing= (
    data
    .withColumn('coast_year_month', F.concat(F.col('coast_name'), F.lit('_'), F.date_format('datetime', 'yyyy-MM')))
)

coast_year_month_dict = {row.coast_year_month: 0.2 for row in data_pre_processing.select('coast_year_month').distinct().collect()}
data_sample = (
    data_pre_processing
    .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=0)
    .drop('coast_year_month')
).toPandas()

In [ ]:
features = ['wind_u', 'wind_v', 'wave_u', 'wave_v', 'wave_period_s']

scaler = StandardScaler()
X = scaler.fit_transform(data_sample[features])

In [ ]:
route = 'scaler.pkl'
joblib.dump(scaler, route)